# Lab 8: DBSCAN Clustering
**Author:** Aditya Narayan Ray


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

print("Aditya Narayan Ray")


In [ ]:
df = pd.read_csv("loanprediction.csv")
print("Dataset shape:", df.shape)
print(df.head())
print(df.info())


In [ ]:
features = [
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "Loan_Amount_Term",
    "Credit_History"
]
X = df[features].apply(pd.to_numeric, errors="coerce")


In [ ]:
imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(X)
print("Missing values handled successfully")


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)
distances = np.sort(distances[:, -1])
plt.figure(figsize=(8, 5))
plt.plot(distances)
plt.xlabel("Data Points")
plt.ylabel("5th Nearest Neighbor Distance")
plt.title("DBSCAN K-Distance Graph")
plt.grid()
plt.show()


In [ ]:
dbscan = DBSCAN(
    eps=0.5,
    min_samples=5
)
labels = dbscan.fit_predict(X_scaled)
df["DBSCAN_Cluster"] = labels
print("DBSCAN applied successfully")


In [ ]:
print("Number of clusters:", len(set(labels)) - (1 if -1 in labels else 0))
print("Number of outliers:", list(labels).count(-1))
print("\nCluster distribution:")
print(df["DBSCAN_Cluster"].value_counts().sort_index())


In [ ]:
plt.figure(figsize=(8, 5))
df["DBSCAN_Cluster"].value_counts().sort_index().plot(
    kind="bar"
)
plt.xlabel("Cluster (-1 = Noise)")
plt.ylabel("Number of Records")
plt.title("DBSCAN Cluster Distribution")
plt.show()


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=labels,
    s=35,
    alpha=0.75
)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("DBSCAN Clustering using PCA")
plt.colorbar(scatter, label="Cluster")
plt.show()
